In [1]:
import numpy as np
import torch
import torch.optim as optim
from dataloaders import GetSingleStepDataLoaders
import model
from train import train_model

In [ ]:
batch_size = 32
output_folder = "./output_cogging/"
save_results = True
use_GPU = True
latent_size = 256
dropout=0.3
epochs = 200

In [3]:
data  = np.load('/local/scratch/groves/jax-forgeRL/models/forging_autoencoder/data/test_comb_FOR_large.npz')
c_t = data['coords_t']
c_tp1 = data['coords_tp1']
steps = data['steps']
positions = data['positions']
rotations = data['rotations']
# actions = np.hstack((steps, positions, rotations))
actions = steps


In [4]:
print(c_t.shape)
print(c_tp1.shape)
print(actions.shape)

(190922, 2048, 3)
(190922, 2048, 3)
(190922, 1)


In [5]:
point_size = c_t.shape[1]

In [6]:
train_loader, test_loader = GetSingleStepDataLoaders(
    coords_t=c_t,       
    coords_tp1=c_tp1,
    actions=actions,
    batch_size=batch_size
)

In [ ]:
# net = model.PCTransitionModel(point_size, latent_size)

net = model.ImprovedPCTransitionModel(
    point_size=point_size,
    latent_size=latent_size,
    dropout=dropout
)

if(use_GPU):
    device = torch.device("cuda:0")
    if torch.cuda.device_count() > 1:
        net = torch.nn.DataParallel(net)
else:
    device = torch.device("cpu")

net = net.to(device)

# optimizer = optim.Adam(net.parameters(), lr=0.0005)

optimizer = torch.optim.Adam(
    net.parameters(),
    lr=0.002,            # Higher than before (was 1e-6!)
    weight_decay=1e-4    # L2 regularization
)

In [8]:
train_model(train_loader, test_loader, net, epochs, optimizer, device, save_results, output_folder)

KeyboardInterrupt: 

In [ ]:
torch.save(net.state_dict(), './transition_model_comb_FOR_large.pth')